In [1]:
!pip install nltk scikit-learn pandas
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [2]:
from google.colab import files
uploaded = files.upload()  # JSON file upload karo

Saving Ecommerce_FAQ_Chatbot_dataset.json to Ecommerce_FAQ_Chatbot_dataset.json


In [3]:
import json
import pandas as pd

with open('Ecommerce_FAQ_Chatbot_dataset.json', 'r') as f:
    data = json.load(f)

# Pehle structure dekho
print(data)

{'questions': [{'question': 'How can I create an account?', 'answer': "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process."}, {'question': 'What payment methods do you accept?', 'answer': 'We accept major credit cards, debit cards, and PayPal as payment methods for online orders.'}, {'question': 'How can I track my order?', 'answer': "You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment."}, {'question': 'What is your return policy?', 'answer': 'Our return policy allows you to return products within 30 days of purchase for a full refund, provided they are in their original condition and packaging. Please refer to our Returns page for detailed instructions.'}, {'question': 'Can I cancel my order?', 'answer': 'You can cancel your order if it has not been shipped yet. Please c

In [4]:
import json
import pandas as pd

with open('Ecommerce_FAQ_Chatbot_dataset.json', 'r') as f:
    data = json.load(f)

df = pd.DataFrame(data['questions'])  # 'questions' key ke andar ki list nikal ke DataFrame banaya
print(df.shape)   # (79, 2) aana chahiye
df.head()


(79, 2)


,question,answer
0,How can I create an account?,"To create an account, click on the 'Sign Up' b..."
1,What payment methods do you accept?,"We accept major credit cards, debit cards, and..."
2,How can I track my order?,You can track your order by logging into your ...
3,What is your return policy?,Our return policy allows you to return product...
4,Can I cancel my order?,You can cancel your order if it has not been s...


In [5]:
!pip install nltk
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df['clean_question'] = df['question'].apply(clean_text)
df[['question', 'clean_question']].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,question,clean_question
0,How can I create an account?,how can i create an account
1,What payment methods do you accept?,what payment methods do you accept
2,How can I track my order?,how can i track my order
3,What is your return policy?,what is your return policy
4,Can I cancel my order?,can i cancel my order


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
question_vectors = vectorizer.fit_transform(df['clean_question'])

def get_answer(user_question):
    clean_q = clean_text(user_question)
    user_vector = vectorizer.transform([clean_q])
    similarity = cosine_similarity(user_vector, question_vectors)
    best_match_idx = similarity.argmax()
    score = similarity[0][best_match_idx]

    if score < 0.2:
        return "Sorry, I don't have an answer for that."
    return df['answer'][best_match_idx]

In [7]:
print(get_answer("how do I make an account"))
print(get_answer("what is the refund process"))
print(get_answer("tell me about weather"))  # ye "no answer" dena chahiye

To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.
If you receive the wrong item in your order, please contact our customer support team immediately. We will arrange for the correct item to be shipped to you and assist with returning the wrong item.
Our email newsletter provides updates on new product releases, exclusive offers, and helpful tips related to our products. You can subscribe to our newsletter on our website.


In [8]:
def get_answer(user_question, debug=True):
    clean_q = clean_text(user_question)
    user_vector = vectorizer.transform([clean_q])
    similarity = cosine_similarity(user_vector, question_vectors)
    best_match_idx = similarity.argmax()
    score = similarity[0][best_match_idx]

    if debug:
        print(f"Best match: '{df['question'][best_match_idx]}' | Score: {score:.3f}")

    if score < 0.3:  # threshold badha diya 0.2 se 0.3
        return "Sorry, I don't have an answer for that."
    return df['answer'][best_match_idx]

In [9]:
get_answer("how do I make an account")
get_answer("what is the refund process")
get_answer("tell me about weather")

Best match: 'How can I create an account?' | Score: 0.740
Best match: 'What should I do if I receive the wrong item?' | Score: 0.442
Best match: 'What is your email newsletter about?' | Score: 0.363


'Our email newsletter provides updates on new product releases, exclusive offers, and helpful tips related to our products. You can subscribe to our newsletter on our website.'

In [10]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]  # ye line add ki
    return ' '.join(words)

In [11]:
# Step 1: clean_question dobara banao naye function se
df['clean_question'] = df['question'].apply(clean_text)

# Step 2: vectorizer dobara fit karo
vectorizer = TfidfVectorizer()
question_vectors = vectorizer.fit_transform(df['clean_question'])

# Step 3: test dobara karo
print(get_answer("how do I make an account"))
print("---")
print(get_answer("what is the refund process"))
print("---")
print(get_answer("tell me about weather"))

Best match: 'How can I create an account?' | Score: 0.674
To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.
---
Best match: 'How can I create an account?' | Score: 0.000
Sorry, I don't have an answer for that.
---
Best match: 'How can I create an account?' | Score: 0.000
Sorry, I don't have an answer for that.


In [ ]:
while True:
    user_input = input("Ask a question (type 'quit' to stop): ")
    if user_input.lower() == 'quit':
        break
    print("Bot:", get_answer(user_input, debug=False))
    print()

Ask a question (type 'quit' to stop): How do I sign up?
Bot: Sorry, I don't have an answer for that.

Ask a question (type 'quit' to stop): What payment options are available?
Bot: We accept major credit cards, debit cards, and PayPal as payment methods for online orders.

Ask a question (type 'quit' to stop): Can I cancel an order after placing it?
Bot: You can cancel your order if it has not been shipped yet. Please contact our customer support team with your order details, and we will assist you with the cancellation process.

Ask a question (type 'quit' to stop): Can I cancel an order after placing it?
Bot: You can cancel your order if it has not been shipped yet. Please contact our customer support team with your order details, and we will assist you with the cancellation process.

Ask a question (type 'quit' to stop): Is it safe to enter my card details?
Bot: Yes, you can return a product purchased with a gift card. The refund will be issued in the form of store credit or a new g